# เทรนรอบ 2 — สอนให้รู้จักแคปซูล

ต่อยอดจาก `pillcount-det.pt` ที่เทรนไว้รอบแรก ไม่ได้เริ่มใหม่

**ปัญหาที่จะแก้:** โมเดลรอบแรกนับเม็ดกลมได้แม่น (spread 1) แต่พอเจอแคปซูลมันตีกรอบขนาดเม็ดกลมปูทับตามความยาว — แคปซูล 2 อันถูกนับเป็น 3 เพราะในชุดเทรนมีแต่เม็ดกลมวางราบ

**วิธีแก้:** รวมชุดข้อมูลแคปซูลเข้ากับของเดิม แล้วเทรนต่อจากน้ำหนักเดิม

---

### ⚠️ สองข้อที่ห้ามทำ และ notebook นี้ระวังไว้ให้แล้ว

**1. ห้ามเทรนด้วยแคปซูลอย่างเดียว** โมเดลจะเรียนแคปซูลแล้ว**ลืมเม็ดกลม** (catastrophic forgetting) เซลล์ที่ 4 จึงรวมสองชุดเข้าด้วยกันก่อน

**2. ห้ามเชื่อ mAP ที่ออกมา** ด้วยเหตุผลเดิมจากรอบแรก — เฟรมใน valid มาจากคลิปเดียวกับ train ตัวตัดสินคือเซลล์ที่ 7 ซึ่งเอาโมเดลไปนับ**ภาพถาดจริงของคุณที่มีแคปซูล** ภาพที่รอบแรกนับผิด

**ก่อนเริ่ม:** `Runtime` → `Change runtime type` → **T4 GPU** → Save

## 1. GPU + ติดตั้ง

In [ ]:
import subprocess

out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", out.stdout.strip() if out.returncode == 0 and out.stdout.strip()
      else "ไม่มี -- Runtime > Change runtime type > T4 GPU")

%pip install -q ultralytics roboflow
import ultralytics
print("ultralytics", ultralytics.__version__)
import glob
import os
import cv2


## 2. API key

พิมพ์ key ลงช่องที่ขึ้นมา — `getpass` จะไม่แสดงตัวอักษรและไม่เก็บลง notebook เพราะงั้นแชร์ notebook นี้ได้โดยไม่หลุด key

In [ ]:
from getpass import getpass

API_KEY = getpass("Roboflow API key: ").strip()
print("รับ key แล้ว ความยาว", len(API_KEY))

## 3. โหลดสองชุดข้อมูลจาก Roboflow

ทั้งคู่เป็น CC BY 4.0

| ชุด | ภาพ | กล่อง | ต่อภาพ |
|---|---|---|---|
| `testpills/pill-count` v4 — เม็ดกลม ถาดน้ำเงิน | 266 | 21,022 | 79 |
| `apisits-workspace-mffve/detection-medicine-capsule` — แคปซูล ถาดขาว | 346 | 20,354 | 59 |

พื้นหลังคนละสีเป็นเรื่องดี — โมเดลที่เห็นทั้งสองแบบจะเรียน**รูปทรง** แทนที่จะจำสีพื้น

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=API_KEY)

# ทุกชุดเป็น CC BY 4.0 ตัวเลขคือที่ดึงจาก API มาตรวจแล้ว
SOURCES = [
    # เม็ดกลม ถาดน้ำเงิน -- ชุดตั้งต้นของเรา
    ("testpills", "pill-count", 4),                      # 266 ภาพ 21,022 กล่อง
    # แคปซูลสีเดียว ถาดขาว -- ชุดที่ทำให้ v2 รู้จักทรงยาว
    ("apisits-workspace-mffve", "detection-medicine-capsule", 4),   # 346 / 20,354
    # เม็ดกลม + แคปซูล แยกคลาส -- ตรงกับโจทย์ 'ยาหลายแบบปนกัน'
    ("kasetsart-university-rpmpb", "pills-pills", None),  # 1,659 / 50,752
    # แคปซูลสองสี -- จุดที่ v2 ยังพลาด
    ("tes-4ynfo", "capsule-mifje", None),                 # 864 / 58,770
]


def latest_version(proj):
    """เลขเวอร์ชันล่าสุด -- v.version เป็นสตริงเต็ม 'ws/proj/3' ไม่ใช่ตัวเลข"""
    nums = []
    for v in proj.versions():
        tail = str(v.version).rstrip("/").split("/")[-1]
        if tail.isdigit():
            nums.append(int(tail))
    return max(nums) if nums else 1


def fetch(workspace, project, version):
    proj = rf.workspace(workspace).project(project)
    if version is None:
        version = latest_version(proj)
        print(f"   ใช้เวอร์ชันล่าสุด = {version}")
    for fmt in ("yolov11", "yolov8"):
        try:
            return proj.version(version).download(fmt).location
        except Exception as exc:                        # noqa: BLE001
            print(f"   {fmt} ไม่ได้: {exc}")
    raise SystemExit(f"โหลด {project} v{version} ไม่สำเร็จ")


downloaded = []
for workspace, project, version in SOURCES:
    print(f"{project} ...")
    loc = fetch(workspace, project, version)
    downloaded.append(loc)
    print("   ->", loc)


## 3b. ดูของก่อนเทรน

**อย่าข้ามเซลล์นี้** ภาพ icon บนหน้าเว็บ Roboflow เป็นภาพสต็อก ไม่ใช่ภาพในชุดจริง — ชุดที่ชื่อว่า capsule อาจเป็นแคปซูลเม็ดเดียวถ่ายใกล้ ซึ่งไม่ช่วยอะไรกับถาดที่มียาหกสิบเม็ด

ดูสองอย่าง: **หน้าตาเหมือนถาดคุณไหม** และ **กรอบครอบแคปซูลทั้งอัน หรือครอบครึ่งเดียว** — ถ้าชุดไหนติดป้ายครึ่งเดียว มันจะสอนโมเดลให้ทำผิดแบบที่เรากำลังจะแก้พอดี


In [ ]:
import random

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(downloaded), figsize=(7 * len(downloaded), 7))
axes = [axes] if len(downloaded) == 1 else list(axes)
for ax, root in zip(axes, downloaded):
    imgs = sorted(glob.glob(f"{root}/train/images/*"))
    if not imgs:
        ax.set_title(f"{os.path.basename(root)}: ไม่มีภาพ")
        ax.axis("off")
        continue
    path = imgs[len(imgs) // 2]
    im = cv2.imread(path)
    h, w = im.shape[:2]
    lab = f"{root}/train/labels/" + os.path.splitext(os.path.basename(path))[0] + ".txt"
    n = 0
    if os.path.isfile(lab):
        for row in open(lab).read().splitlines():
            f = row.split()
            if len(f) < 5:
                continue
            n += 1
            cx, cy, bw, bh = (float(v) for v in f[1:5])
            cv2.rectangle(im, (int((cx - bw / 2) * w), int((cy - bh / 2) * h)),
                          (int((cx + bw / 2) * w), int((cy + bh / 2) * h)),
                          (90, 220, 110), 2)
    ax.imshow(im[:, :, ::-1])
    ax.set_title(f"{os.path.basename(root)}\n{n} กล่องในภาพนี้", fontsize=12)
    ax.axis("off")
plt.tight_layout()
plt.show()

for root in downloaded:
    imgs = glob.glob(f"{root}/train/images/*")
    boxes = sum(len([r for r in open(t).read().splitlines() if r.strip()])
                for t in glob.glob(f"{root}/train/labels/*.txt"))
    print(f"{os.path.basename(root):<44} {len(imgs):>5} ภาพ  {boxes:>7} กล่อง  "
          f"{boxes / max(len(imgs), 1):>6.1f} ต่อภาพ")


## 4. รวมเป็นชุดเดียว คลาสเดียว

ชุดแคปซูลมีชื่อคลาสของมันเอง ชุดเราใช้ `pill` — เซลล์นี้ยุบทุกคลาสเป็น `0 = pill`

**ทำไมคลาสเดียว:** บั๊กที่จะแก้คือ "แคปซูล 1 อัน ได้ 2 กรอบ" ซึ่งเป็นเรื่อง**รูปทรง** คลาสเดียวสอนได้แล้ว การแยก `tablet` / `capsule` เป็นคนละโจทย์ ค่อยทำทีหลังเมื่อนับถูกแล้ว

In [ ]:
import glob
import os
import shutil

MERGED = "/content/merged"
shutil.rmtree(MERGED, ignore_errors=True)
for split in ("train", "valid", "test"):
    os.makedirs(f"{MERGED}/{split}/images", exist_ok=True)
    os.makedirs(f"{MERGED}/{split}/labels", exist_ok=True)

tally = {}
for src_i, root in enumerate(downloaded):
    tag = f"s{src_i}"                       # กันชื่อไฟล์ชนกันระหว่างสองชุด
    for split in ("train", "valid", "test"):
        for img in glob.glob(f"{root}/{split}/images/*"):
            stem = os.path.splitext(os.path.basename(img))[0]
            lab = f"{root}/{split}/labels/{stem}.txt"
            if not os.path.isfile(lab):
                continue                    # ภาพที่ไม่มี label ทิ้งไป
            ext = os.path.splitext(img)[1]
            shutil.copy(img, f"{MERGED}/{split}/images/{tag}_{stem}{ext}")
            # ทุกคลาสกลายเป็น 0
            lines = []
            for row in open(lab).read().splitlines():
                parts = row.split()
                if len(parts) >= 5:
                    lines.append(" ".join(["0"] + parts[1:]))
            with open(f"{MERGED}/{split}/labels/{tag}_{stem}.txt", "w") as fh:
                fh.write("\n".join(lines) + ("\n" if lines else ""))
            tally[(tag, split)] = tally.get((tag, split), 0) + 1

DATA_YAML = f"{MERGED}/data.yaml"
with open(DATA_YAML, "w") as fh:
    fh.write(f"path: {MERGED}\ntrain: train/images\nval: valid/images\n"
             "test: test/images\nnc: 1\nnames: ['pill']\n")

print("ที่มาของภาพ:", {f"{k[0]}/{k[1]}": v for k, v in sorted(tally.items())})
for split in ("train", "valid", "test"):
    imgs = glob.glob(f"{MERGED}/{split}/images/*")
    boxes = sum(len([r for r in open(t).read().splitlines() if r.strip()])
                for t in glob.glob(f"{MERGED}/{split}/labels/*.txt"))
    print(f"{split:<6} {len(imgs):>5} ภาพ  {boxes:>7} กล่อง  "
          f"{boxes / max(len(imgs), 1):>6.1f} ต่อภาพ")

## 5. อัปโหลดน้ำหนักเดิม และภาพทดสอบ

ทั้ง 4 ไฟล์อยู่ในโฟลเดอร์เดียวกันแล้ว: `D:\pillsort\model3\upload\`

กด `Choose Files` → เข้าโฟลเดอร์นั้น → **Ctrl+A** → Open

| ไฟล์ | คืออะไร |
|---|---|
| `pillcount-det-v2.pt` | น้ำหนักตัวที่ดีที่สุดตอนนี้ — **จะเทรนต่อจากตัวนี้** |
| `test_capsules_only_true4.jpg` | ถาดแคปซูลล้วน **ของจริง 4 อัน** v2 นับได้ 6 |
| `test_capsule_in_pile.jpg` | แคปซูลแทรกในกองยา **ของจริง 23** v2 นับได้ 24 |
| `test_tablets_true61.jpg` | เม็ดกลมล้วน **ของจริง ~61** v2 นับถูก — ต้องไม่แย่ลง |

เซลล์นี้รันซ้ำได้ ถ้าเลือกไม่ครบก็เลือกเพิ่มทีหลัง ของเดิมไม่หาย


In [ ]:
from google.colab import files

BENCH = "/content/bench"
os.makedirs(BENCH, exist_ok=True)

# globals() ไม่ใช่การอวดรู้ -- เซลล์นี้ถูกรันซ้ำได้ ถ้าเลือกไฟล์ทีละอัน
# แล้วตั้ง START_WEIGHTS = None ทุกครั้ง .pt ที่อัปรอบก่อนจะหายไปเงียบๆ
START_WEIGHTS = globals().get("START_WEIGHTS")

for fname, blob in files.upload().items():
    if fname.endswith(".pt"):
        START_WEIGHTS = f"/content/{fname}"
        open(START_WEIGHTS, "wb").write(blob)
    else:
        open(os.path.join(BENCH, fname), "wb").write(blob)

bench = sorted(glob.glob(f"{BENCH}/*"))
print()
print("น้ำหนักเริ่มต้น:", START_WEIGHTS or "ยังไม่มี")
print("ภาพทดสอบ  :", [os.path.basename(p) for p in bench] or "ยังไม่มี")

if START_WEIGHTS is None or len(bench) < 2:
    print()
    print("ยังไม่ครบ -- รันเซลล์นี้ซ้ำแล้วเลือกไฟล์ที่ขาด ของเดิมไม่หาย")
else:
    print()
    print("ครบแล้ว ไปเซลล์ถัดไปได้")


## 6. เทรนต่อ

เริ่มจาก `pillcount-det.pt` ไม่ใช่ `yolo11n.pt` — ของเดิมนับเม็ดกลมได้ spread 1 แล้ว ทิ้งไปเริ่มใหม่คือทิ้งของดี

epochs น้อยกว่ารอบแรก (60 ไม่ใช่ 100) เพราะไม่ได้เริ่มจากศูนย์ และ `lr0` ต่ำลงเพื่อไม่ให้ไปรื้อสิ่งที่มันรู้อยู่แล้วแรงเกินไป

In [ ]:
from ultralytics import YOLO

# ไม่มีทางสำรองอีกแล้ว รอบแรกมันตกมาใช้ yolo11n.pt เงียบๆ แล้วเสีย GPU ไป 15 นาที
# กว่าจะรู้ก็ตอนเปิด train_args ของไฟล์ที่โหลดกลับมาแล้ว
if START_WEIGHTS is None or not os.path.isfile(START_WEIGHTS):
    raise SystemExit(
        "ยังไม่ได้อัปโหลด pillcount-det-v2.pt -- กลับไปรันเซลล์ที่ 5 ก่อน\n"
        "จุดประสงค์ของรอบนี้คือเทรนต่อจากน้ำหนักเดิม ถ้าเริ่มจาก yolo11n.pt "
        "ก็เป็นการเทรนใหม่ ซึ่งเป็นคนละการทดลอง")

print("เทรนต่อจาก:", START_WEIGHTS)
model = YOLO(START_WEIGHTS)
results = model.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,
    batch=16,
    patience=20,
    lr0=0.005,            # ครึ่งหนึ่งของค่าปกติ: ปรับจูน ไม่ใช่รื้อสร้างใหม่
    project="/content/runs",
    name="pillcount-v3",
    plots=True,
)

# ยืนยันทันทีว่ามันเริ่มจากที่ควรเริ่มจริง ไม่ต้องรอไปเปิดดูทีหลัง
import torch

best = "/content/runs/pillcount-v3/weights/best.pt"
ck = torch.load(best, map_location="cpu", weights_only=False)
print("\nweights:", best)
print("เริ่มจาก:", ck.get("train_args", {}).get("model"))

## 7. ข้อสอบจริง — สามข้อ

คะแนน mAP จากเซลล์ที่ 6 **เชื่อไม่ได้** เฟรมใน valid มาจากคลิปเดียวกับ train ตัวตัดสินคือสามภาพนี้ ซึ่งไม่มีอะไรเคยเทรนบนมันเลย

| ภาพ | v2 ทำได้ | v3 ต้องได้ |
|---|---|---|
| แคปซูลล้วน (จริง 4) | 6 ✗ | **4** |
| แคปซูลในกองยา (จริง 23) | 24 ✗ | **23** |
| เม็ดกลมล้วน (จริง ~61) | 61 ✓ | **61 — ห้ามเพี้ยน** |

แถวสุดท้ายสำคัญที่สุด ถ้าเลขนั้นเปลี่ยนไปมากแปลว่าโมเดล**ลืมของเดิม** ให้ลด `lr0` ลงครึ่งหนึ่งแล้วเทรนใหม่

`อัตราส่วนกรอบสูงสุด` คือตัวบอกว่ามันรู้จักทรงยาวแค่ไหน — ถ้าแคปซูลได้กรอบเดียวทั้งอัน ค่านี้จะขึ้นไปแถว 2.0–3.0


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

TRUTH = {"test_capsules_only_true4": 4, "test_capsule_in_pile": 23,
         "test_tablets_true61": 61}

trained = YOLO(best)

if not bench:
    print("ไม่มีภาพทดสอบ -- กลับไปรันเซลล์ที่ 5")
else:
    fig, axes = plt.subplots(1, len(bench), figsize=(8 * len(bench), 7))
    axes = [axes] if len(bench) == 1 else list(axes)
    for ax, path in zip(axes, bench):
        stem = os.path.splitext(os.path.basename(path))[0]
        r = trained.predict(path, conf=0.45, iou=0.4, imgsz=640, max_det=1000,
                            verbose=False)[0]
        b = r.boxes.xyxy.cpu().numpy()
        if len(b):
            w, h = b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]
            ar = np.maximum(w / h, h / w)
        else:
            ar = np.array([0.0])
        truth = TRUTH.get(stem)
        verdict = ""
        if truth:
            verdict = " ตรง" if len(b) == truth else f" ต่าง {len(b) - truth:+d}"
        ax.imshow(r.plot(labels=False, line_width=2)[:, :, ::-1])
        ax.set_title(f"{stem}\n{len(b)} เม็ด" +
                     (f"  (จริง {truth}){verdict}" if truth else ""), fontsize=13)
        ax.axis("off")
        print(f"{stem:<30} {len(b):>4} เม็ด   จริง {str(truth or '?'):>3}"
              f"{verdict:<10} อัตราส่วนกรอบสูงสุด {ar.max():.2f}")
    plt.tight_layout()
    plt.show()


## 8. โหลดกลับเครื่อง

เอาไปวางที่ `D:\pillsort\model3\weights\` **อย่าทับตัวเดิม** — ตั้งชื่อ `pillcount-det-v2.pt` เพื่อให้เทียบสองตัวกันได้บนม้านั่งจริง ตัวเก่ามีตัวเลขที่วัดไว้แล้ว (spread 1 บนคลิป, spread 3 บนกล้อง) ถ้าทับทิ้งก็ไม่เหลืออะไรให้เทียบ

In [ ]:
shutil.copy(best, "/content/pillcount-det-v3.pt")
files.download("/content/pillcount-det-v3.pt")